# ⚙️ Google Colab Setup (added for portability)

> **This section was added so the tutorial runs on Google Colab.** Run the STEP 1–5 cells
> below **once, top to bottom**, then continue with the original tutorial cells underneath.
>
> The original notebook was written on the instructor's server and hard-codes paths like
> `/home/ye/exp/SMT-NMT_tutorial` and `/home/ye/tool/...`. Colab runs as **root**, so these
> cells **recreate those exact directories**, which means the original cells work with
> no path edits needed.

### 📖 Dataset change: Thai ➜ Rakhine (please read)

The original class used a **Myanmar–Thai** (`.my`/`.th`) medical parallel corpus that is
**not public**. This Colab version instead uses the **Myanmar–Rakhine** (`.my`/`.rk`)
parallel corpus released by **Sayar Ye Kyaw Thu** in the public
[MTRSS repo](https://github.com/ye-kyaw-thu/MTRSS) (Machine Translation Research Summer
School, YTU) — conveniently the **same splits**: 5,000 train / 500 dev / 100 test pairs.

Consequences you will see throughout the notebook:
- Data files are `train.rk`, `dev.rk`, `test.rk` (real extension, no disguising).
- The Moses pipeline **auto-detects** language codes from the file extensions, so the
  experiment folders become `my-rk/` and `rk-my/` (instead of `my-th/` / `th-my/`).
- Every original cell that referenced `th`, `my-th`, or `th-my` has been updated to
  `rk`, `my-rk`, `rk-my` — the *logic* is 100% unchanged.
- Rakhine (Arakanese) is **closely related to Burmese**, so expect a much **higher BLEU**
  than the 11.77 the instructor got for Thai–Myanmar. Same pipeline, easier language pair.

### What the setup does
1. **System packages** — `perl`, `tree`, build tools, Boost, `IPC::System::Simple`.
2. **Asset files** — recreates the scripts, `syl_normalizer.py`, configs, and tools
   (pulled from the public [MTRSS](https://github.com/ye-kyaw-thu/MTRSS) and
   [sylbreak](https://github.com/ye-kyaw-thu/sylbreak) repos, with all author paths rewritten).
3. **Moses SMT** — **builds the decoder from source** (the official prebuilt binaries are no longer downloadable). ⏳ ~20–40 min, run once.
4. **GIZA++** — clones and builds the word aligner, and applies the notebook's `mkcls` fix.
5. **Corpus + dictionary + images** — downloads the Myanmar–Rakhine corpus, generates a
   syllable dictionary, writes placeholder images, then `cd`s into the tutorial root.

### ⚠️ Remaining honest caveats
- **`USER=root` is required.** GIZA++ crashes (segfault) if the `USER` environment
  variable is unset — which is the case on Colab. STEP 1 and STEP 5 both set it;
  if you ever restart the runtime and skip them, run `%env USER=root` before training.
- **Syllable dictionary is generated** from the Burmese training side (the original
  `final_syl_dictionary_13Feb2024.sorted.txt` is not public).
- **Images are placeholders** (the instructor's screenshots aren't shipped with the notebook).
- **Moses is the one fragile dependency on modern Colab.** The official prebuilt binaries
  are gone (server returns 403), so STEP 3 **compiles Moses from source (~20–40 min)**. If it
  fails, just re-run STEP 3 (the build resumes) — see the STEP 3b troubleshooting cell.
- This notebook contains the instructor's **deliberate error → fix walkthrough**, so a strict
  "Run all" may still reproduce those teaching errors by design.


In [ ]:
#@title ⚙️ STEP 1 — System dependencies { display-mode: "form" }
import os, sys
assert ("google.colab" in sys.modules) or os.path.exists("/content"), \
    "This setup targets Google Colab."
os.chdir("/content")

# CRITICAL: GIZA++ calls getenv("USER") and strlen()'s the result without a
# NULL check (file_spec.h:53). Colab does not set USER, so without this line
# GIZA++ segfaults instantly and EMS dies at TRAINING:run-giza with no
# evaluation output. Verified by gdb backtrace in a clean Ubuntu 22.04 test.
os.environ["USER"] = "root"
print("Installing system packages (this takes a minute)...")
!apt-get -qq update > /tmp/apt.log 2>&1
!apt-get -qq install -y tree perl build-essential cmake automake libtool \
    imagemagick graphviz \
    libboost-all-dev libbz2-dev liblzma-dev zlib1g-dev libgoogle-perftools-dev \
    libipc-system-simple-perl wget curl git >> /tmp/apt.log 2>&1
!pip -q install pillow >> /tmp/apt.log 2>&1
print("STEP 1 done. (full log: /tmp/apt.log)")


In [ ]:
#@title ⚙️ STEP 2 — Recreate directory layout & write asset files { display-mode: "form" }
import os, json, base64

TUT  = "/home/ye/exp/SMT-NMT_tutorial"
TOOL = "/home/ye/tool"
for d in [TUT, TUT+"/data", TUT+"/scripts", TUT+"/syl_normalizer", TUT+"/tool",
          TUT+"/error-examples", TUT+"/pbsmt", TOOL, "/home/lar/tool"]:
    os.makedirs(d, exist_ok=True)

EMBEDDED_B64 = "..."
FILES = json.loads(base64.b64decode(EMBEDDED_B64).decode("utf-8"))
for rel, content in FILES.items():
    p = os.path.join(TUT, rel)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        f.write(content)
    if rel.rsplit(".", 1)[-1] in ("pl", "sh", "py"):
        os.chmod(p, 0o755)
print(f"Wrote {len(FILES)} asset files under {TUT}")
for rel in sorted(FILES):
    print("  ", rel)


In [ ]:
#@title ⚙️ STEP 3 — Build Moses SMT from source (~20–40 min, run once) { display-mode: "form" }
# NOTE: the official prebuilt Moses binaries (statmt.org) are no longer downloadable
# (the server returns HTTP 403), so we build Moses from source. It's slow but reliable.
# You only need to do this ONCE per Colab session. If it stops/disconnects, just re-run
# this cell — bjam resumes where it left off.
import os, multiprocessing

MOSES = "/home/ye/tool/mosesbin/ubuntu-17.04/moses"   # keep the exact path the notebook expects
os.makedirs(os.path.dirname(MOSES), exist_ok=True)

print("1/3  Installing build dependencies ...")
!apt-get -qq update >> /tmp/apt.log 2>&1
!apt-get -qq install -y build-essential git pkg-config automake libtool \
    libboost-all-dev libbz2-dev liblzma-dev zlib1g-dev libgoogle-perftools-dev \
    libcmph-dev cmake >> /tmp/apt.log 2>&1

if not os.path.exists(MOSES + "/bin/moses"):
    if not os.path.isdir(MOSES + "/.git"):
        !rm -rf {MOSES}
        print("2/3  Cloning mosesdecoder ...")
        !git clone -q --depth 1 https://github.com/moses-smt/mosesdecoder.git {MOSES}
    os.chdir(MOSES)
    n = multiprocessing.cpu_count()
    print(f"3/3  Compiling with {n} cores — grab a coffee, this takes ~20–40 min ...")
    # Build the decoder + KenLM (lmplz) + tuning tools against the system Boost.
    # We omit --with-xmlrpc-c on purpose so the optional network server is skipped
    # (that module is the most fragile to build and the EMS pipeline does not need it).
    !./bjam --with-boost=/usr --with-cmph=/usr -j{n} 2>&1 | tail -25
else:
    print("Moses already built — skipping.")

# ---- verify ----
ok_moses = os.path.exists(MOSES + "/bin/moses")
ok_lmplz = os.path.exists(MOSES + "/bin/lmplz")
ok_train = os.path.exists(MOSES + "/scripts/training/train-model.perl")
print("\n— Build result —")
print("  moses decoder    :", "✅" if ok_moses else "❌", MOSES + "/bin/moses")
print("  lmplz (KenLM)    :", "✅" if ok_lmplz else "❌")
print("  train-model.perl :", "✅" if ok_train else "❌")
if ok_moses and ok_lmplz:
    print("\n✅ Moses is ready. Continue to STEP 4.")
else:
    print("\n⚠️ Build incomplete — read the last lines above for the actual error,")
    print("   then see the STEP 3b troubleshooting cell below.")


## ⚙️ STEP 3b — If the Moses build failed (troubleshooting)

Building older Moses on the newest Colab Ubuntu is the **one genuinely fragile part** of
this setup. If STEP 3 ended with ❌, try these in order:

1. **Just re-run STEP 3.** `bjam` is incremental — re-running resumes the build and often
   pushes past a transient hiccup or a step that ran out of time before a disconnect.

2. **Check the real error.** STEP 3 prints the last 25 lines of the build log. The full log
   of a fresh attempt helps — run this to see more:
   ```python
   !cd /home/ye/tool/mosesbin/ubuntu-17.04/moses && ./bjam --with-boost=/usr --with-cmph=/usr -j2 2>&1 | tail -60
   ```

3. **Force a clean rebuild** (if a half-built state is stuck):
   ```python
   !cd /home/ye/tool/mosesbin/ubuntu-17.04/moses && ./bjam --clean -a 2>/dev/null; \
    cd /home/ye/tool/mosesbin/ubuntu-17.04/moses && ./bjam --with-boost=/usr --with-cmph=/usr -j2 -a 2>&1 | tail -40
   ```

4. **What you actually need.** The pipeline only needs `bin/moses`, `bin/lmplz`,
   `bin/build_binary`, the `mert` tools, and the `scripts/` folder (the Perl scripts ship
   with the source and don't need compiling). If those exist, ignore other build warnings.

> The rest of the setup (GIZA++, corpus, scripts, paths, Python tools) does **not** depend
> on this and will work regardless.


In [ ]:
#@title ⚙️ STEP 4 — Build GIZA++ word aligner { display-mode: "form" }
import os
GIZA = "/home/ye/tool/giza-pp"
if not os.path.exists(GIZA + "/GIZA++-v2/GIZA++"):
    os.chdir("/home/ye/tool")
    !git clone -q https://github.com/moses-smt/giza-pp
    os.chdir(GIZA)
    !make all -j2 2>&1 | tail -8
# Notebook's known fix: GIZA++ looks for mkcls inside GIZA++-v2/
!cp -f /home/ye/tool/giza-pp/mkcls-v2/mkcls /home/ye/tool/giza-pp/GIZA++-v2/ 2>/dev/null
print("GIZA++ built:", os.path.exists(GIZA + "/GIZA++-v2/GIZA++"),
      "| mkcls present:", os.path.exists(GIZA + "/GIZA++-v2/mkcls"))


In [ ]:
#@title ⚙️ STEP 5 — Stage corpus, dictionary, images & finalize { display-mode: "form" }
import os, sys, glob, subprocess, collections, urllib.request
os.environ["USER"] = "root"   # GIZA++ segfaults if USER is unset (see STEP 1)

TUT = "/home/ye/exp/SMT-NMT_tutorial"

# 1) Parallel corpus: Myanmar-Rakhine from Sayar Ye Kyaw Thu's public MTRSS repo
#    (same splits as the original class corpus: 5000 train / 500 dev / 100 test)
print("Downloading Myanmar-Rakhine parallel corpus (ye-kyaw-thu/MTRSS)...")
base = "https://raw.githubusercontent.com/ye-kyaw-thu/MTRSS/master/pbsmt/data/"
for split in ["train", "dev", "test"]:
    for lang in ["my", "rk"]:
        urllib.request.urlretrieve(base + f"{split}.{lang}", f"{TUT}/data/{split}.{lang}")
# normalize line endings
for f in glob.glob(TUT + "/data/*"):
    b = open(f, "rb").read().replace(b"\r\n", b"\n").replace(b"\r", b"\n")
    open(f, "wb").write(b)
for f in sorted(glob.glob(TUT + "/data/*")):
    print("   ", os.path.basename(f), sum(1 for _ in open(f, encoding="utf-8")), "lines")

# 2) Syllable dictionary generated from the Burmese training side
print("Generating syllable-frequency dictionary from train.my ...")
seg = subprocess.run([sys.executable, f"{TUT}/tool/sylbreak.py",
                      "--input", f"{TUT}/data/train.my", "--separator", " "],
                     capture_output=True, text=True).stdout
cnt = collections.Counter(seg.split())
dict_path = f"{TUT}/syl_normalizer/final_syl_dictionary_13Feb2024.sorted.txt"
with open(dict_path, "w", encoding="utf-8") as f:
    for syl, c in sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0])):
        f.write(f"{syl} {c}\n")
print("   dictionary entries:", len(cnt))

# 3) Placeholder images so the IPython.Image() cells don't crash
from PIL import Image, ImageDraw
for p, label in [
    (f"{TUT}/syl_normalizer/syl_normalizer.png", "syl_normalizer algorithm (placeholder image)"),
    (f"{TUT}/error-examples/GIZA-error.png",     "GIZA++ mkcls error (placeholder image)"),
    (f"{TUT}/error-examples/graph.1.my-rk.png",  "my-rk EMS pipeline graph (placeholder image)"),
    (f"{TUT}/error-examples/graph.1.my-rk-2ndtime.png", "my-rk EMS graph, 2nd run (placeholder image)"),
]:
    img = Image.new("RGB", (820, 180), (245, 245, 247))
    ImageDraw.Draw(img).text((24, 80), label, fill=(70, 70, 80))
    img.save(p)

# 4) Make all scripts executable
for f in glob.glob(TUT + "/scripts/*") + glob.glob(TUT + "/pbsmt/*.pl") + glob.glob(TUT + "/pbsmt/*.sh"):
    os.chmod(f, 0o755)

# 5) Land in the tutorial root (this is the CWD the original cells assume)
os.chdir(TUT)
print("\n✅ Setup complete. Working directory:", os.getcwd())
print("   Corpus: Myanmar (train.my) <-> Rakhine (train.rk)")
print("   Now continue with the original tutorial cells below.")


# Statistical Machine Translation Tutorial (Intro)
Ye Kyaw Thu, Lab Leader, Language Understanding Lab., Myanmar  
For AI Fundamental Class students  
Date: 26 April 2026  

ဒီ Notebook က Github က MTRSS (Machine Translation Research Summer School) က code တွေကို ဆာဗာတစ်ခုပေါ်ကို ဒေါင်းလုဒ်လုပ်ပြီး အတန်းထဲမှာ ပထမဆုံး run ပြခဲ့တာမို့ error သုံးခု တက်တာကိုလည်း လက်တွေ့ရှင်းရင်းနဲ့ run ခဲ့တဲ့ notebook ပါ။  

- ပထမ error က ထုံးစံအတိုင်းပဲ perl script တွေ၊ configuration file မှာ ကိုယ့်စက်ရဲ့ path တွေကို ဖြည့်တဲ့အခါမှာ ကျန်ခဲ့တာမျိုးပါ
- ဒုတိယ error က GIZA++ နဲ့ ပတ်သက်ပြီး ဖြစ်တဲ့ error ပါ။ GIZA++ install လုပ်ထားတဲ့ path (/home/ye/tool/giza-pp/GIZA++-v2) အောက်မှာ mklcls ဖိုင်ကိုပါ ကော်ပီကူးထည့်လိုက်ရင် ပြေလည်းပါတယ်။   
- တတိယ error ကတော့ SGM အတွက် ဆောက်ထားတဲ့ ဖိုလ်ဒါနာမည်ကို စကားပြောရင်း၊ စာသင်ရင်းနဲ့ ရိုက်လိုက်တာမို့ test-gen ဆိုပြီး မှာရိုက်လိုက်လို့ SMT run တဲ့အခါမှာ တက်တဲ့ error ပါ။ အမှန်က test-sgm ဖြစ်ရပါမယ်။ :P

## Data

In [ ]:
%pwd

In [ ]:
!ls ./data/*

## Data Size

for training:  

In [ ]:
!wc ./data/train.*

for development or validation:  

In [ ]:
!wc ./data/dev.*

for testing:  

In [ ]:
!wc ./data/test.*

Let's see data format:  

In [ ]:
!head ./data/train.rk

In [ ]:
!head ./data/train.my

In [ ]:
!head ./data/dev.rk

In [ ]:
!head ./data/dev.my

In [ ]:
!head ./data/test.rk

In [ ]:
!head ./data/test.my

## Data Cleaning

မနေ့ကအတန်းမှာ ရှင်းပြခဲ့သလိုပဲ။ မြန်မာစာကို sylbreak.py နဲ့ syllable breaking လုပ်တဲ့အခါမှာ စာရိုက်ထားတဲ့ အစီအစဉ် (typing order) က မမှန်တာမျိုးတွေကို တွေ့ရတယ်။ ဥပမာ "စွ န့်", "ကြ ည့်"  

အဲဒါမျိုး မဖြစ်အောင် အရင်ဆုံး မြန်မာစာ ဒေတာတွေကို cleaning လုပ်ကြရအောင်။  
မလုပ်ရင် machine translation performance ကို အနည်းနဲ့အများ သက်ရောက်မှုရှိလိမ့်မယ်။  
ပြီးတော့ ရခိုင် ကနေ မြန်မာစာကို ဘာသာပြန်ခိုင်းပြီး ထွက်လာတဲ့ output စာကြောင်းတွေမှာလည်း အမှားတွေပါနေလိမ့်မယ်။  

အရင်ဆုံး ဘယ်လိုမှားနေတယ် ဆိုတာကို မြင်သာအောင် အထက်က မှားနေတဲ့ စာလုံး နှစ်လုံးကိုပဲ သုံးပြီး အမှန် ရိုက်ထားတာနဲ့ နှိုင်းယှဉ်ကြည့်ကြရအောင်။  

text file တစ်ဖိုင်အနေနဲ့ အောက်ပါ စာကြောင်းတွေကို သိမ်းပါ။ အမှားက အပေါ်က၊ မှန်မှန်ကန်ကန် ရိုက်ထည့်ထားတာက အောက်က syllable ပါ။  

စွန့်  
စွန့်  
ကြည့်  
ကြည့်  

In [ ]:
%%bash
cat <<EOF > error-eg.txt
စွန့်
စွန့်
ကြည့်
ကြည့်
EOF

ဖိုင် အဖြစ် သိမ်းမသိမ်းကို confirm လုပ်ကြည့်ပါ။  

In [ ]:
!cat ./error-eg.txt

tool/ folder အောက်မှာ Unicode number တွေအနေနဲ့ print ထုတ်ပေးအောင် ရေးထားတဲ့ perl script ရှိတယ်။ အဲဒါကို သုံးပြီး အထက်က မြင်နေရတဲ့ မြန်မာစာ စာလုံးတွေကို ရိုက်ထုတ်ခိုင်းကြည့်ရအောင်။ အဲဒါဆိုရင် မှားတဲ့ syllable စာလုံးနဲ့ မှန်တဲ့ syllable စာလုံးအကြားမှာ typing order က ဘယ်လိုဖြစ်နေသလဲ ဆိုတာကို မြင်ရပါလိမ့်မယ်။ လေ့လာကြည့်ပါ။   

In [ ]:
!perl ./tool/print-codepoint.pl ./error-eg.txt

## Notes
အထက်မှာ မြင်ရတဲ့အတိုင်းပဲ။ ကိုယ့်ဖိုင်ထဲမှာ ရိုက်ထားတဲ့ syllable တွေရဲ့ order က မှားနေရင်တော့ sylbreak tool နဲ့ syllable segmentation လုပ်တဲ့အခါမှာ မှားဖြတ်ပေးပါလိမ့်မယ်။ pipe ကို delimiter အဖြစ်ထားပြီး ဖြတ်ရင် အောက်ပါလိုမျိုး ရလဒ် ရလိမ့်မယ်...  

In [ ]:
!python ./tool/sylbreak.py --input ./error-eg.txt

## Normalization  

အထက်ပါလိုမျိုး typing order အပြင်ကို တကယ်က spelling mistake, translation error စတာမျိုးတွေလည်း အများကြီး ပါနိုင်ပါတယ်။  
ဒီ tutorial မှာတော့ syllable level အမှားတွေကိုပဲ ပြင်တဲ့အပိုင်း တနည်းအားဖြင့် syllable normalization အပိုင်းကိုပဲ data cleaning ဥပမာ အနေနဲ့ လုပ်ပြပါမယ်။  

အရင်ဆုံး လက်ရှိ data folder တစ်ခုလုံးကို ကောပီကူးပြီး အဲဒီအထဲမှာပဲ normalization process ကို လုပ်ကြရအောင်။   
မဟုတ်ရင် Myanmar-Rakhine parallel corpus ဖိုင်တွေကို မှားပြီး ဖျက်မိတာ၊ ပြင်မိတာမျိုးတွေ ရှိရင် ပြန်ရဖို့ ခက်တာမို့။  

In [ ]:
!cp -r ./data ./clean-data

In [ ]:
!ls ./clean-data/*

In [ ]:
%cd ./clean-data/

In [ ]:
%mkdir normalize

In [ ]:
!mv *.* ./normalize

In [ ]:
!tree

အသစ်ဆောက်ထားတဲ့ normalize/ folder အောက်မှာပဲ မှားနေတဲ့ syllable တွေကို ပြင်တာတွေ လုပ်မယ်။ ပြီးမှ data-clean/ အောက်ကို ကော်ပီပြန်ကူးယူကြမယ်။  

Normalization tool က ဆရာတို့ Lab က develop လုပ်နေတုန်းပါပဲ။ လက်ရှိ ဗားရှင်းကိုပဲ သုံးကြရအောင်။  


In [ ]:
%cd ..

In [ ]:
!ls ./syl_normalizer/

 **syl_normalizer.py code ကို လေ့လာကြည့်ရအောင်။**  

In [ ]:
!cat ./syl_normalizer/syl_normalizer.py

### Algorithm of syl_normalizer

In [ ]:
from IPython import display
display.Image("/home/ye/exp/SMT-NMT_tutorial/syl_normalizer/syl_normalizer.png", width=300)

အထက်က တုန်းက သုံးပြခဲ့တာက 2024 တုန်းက ဗားရှင်းပါ။ 
Notebook 2, 3 မှာက updated လုပ်ထားတဲ့ syl_normalization.py ကို သုံးထားပါတယ်။ GitHub မှာလည်း updated လုပ်ထားတဲ့ ဗားရှင်းကိုပဲ တင်ပေးထားလိုက်ပါမယ်။  
### Let's clean training, development and test dataset of Myanmar language

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/syl_normalizer

In [ ]:
!python ./syl_normalizer.py --help

In [ ]:
!python ./syl_normalizer.py --dictionary ./final_syl_dictionary_13Feb2024.sorted.txt \
--frequency 2 --input ../clean-data/normalize/train.my \
--output ../clean-data/normalize/train.my.clean 

In [ ]:
!wc ../clean-data/normalize/train.my

In [ ]:
!wc ../clean-data/normalize/train.my.clean

In [ ]:
!head ../clean-data/normalize/train.my.clean

In [ ]:
!head ../clean-data/normalize/train.my

In [ ]:
!tail ../clean-data/normalize/train.my.clean

In [ ]:
!tail ../clean-data/normalize/train.my

**Development data cleaning ...**

In [ ]:
!python ./syl_normalizer.py --dictionary ./final_syl_dictionary_13Feb2024.sorted.txt \
--frequency 2 --input ../clean-data/normalize/dev.my \
--output ../clean-data/normalize/dev.my.clean

In [ ]:
!wc ../clean-data/normalize/dev.my

In [ ]:
!wc ../clean-data/normalize/dev.my.clean

**Test data cleaning ...**

In [ ]:
!python ./syl_normalizer.py --dictionary ./final_syl_dictionary_13Feb2024.sorted.txt \
--frequency 2 --input ../clean-data/normalize/test.my \
--output ../clean-data/normalize/test.my.clean

In [ ]:
!wc ../clean-data/normalize/test.my

In [ ]:
!wc ../clean-data/normalize/test.my.clean

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/clean-data/normalize/

In [ ]:
!head ./test.my

In [ ]:
!head ./test.my.clean

**cleaning လုပ်ထားတဲ့ ဖိုင်တွေကို machine translation လုပ်တဲ့အခါမှာ သုံးဖို့အတွက် ကော်ပီကူး ပြင်ထားကြရအောင်**

သတိထားရမှာက — အရင်က `mv *.*` လုပ်တုန်းက **ရခိုင်ဘက် ဖိုင်တွေ (.rk) လည်း** normalize/ ထဲ ပါသွားခဲ့ပါတယ်။ ရခိုင်ဘက်ကို normalize လုပ်စရာ မလိုပေမဲ့ (ဆရာ့ မူရင်း flow မှာလည်း မြန်မာဘက်ပဲ clean လုပ်ပါတယ်)၊ SMT training အတွက် clean-data/ ထဲမှာ ဘာသာစကား **နှစ်ဖက်လုံး** ရှိရမှာမို့ .rk ဖိုင်တွေကိုပါ ပြန်ကူးထည့်ရပါမယ်။


In [ ]:
%pwd

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/clean-data/


In [ ]:
!cp ./normalize/*.clean .
!cp ./normalize/*.rk .


In [ ]:
!tree .

**.clean တွေကို ဖြုတ်ကြရအောင်**  
SMT မှာက နောက်ဆုံး ဖိုင်နာမည်နောက်ဆုံးပိုင်း (i.e. file extension) က source/target ဘာသာစကားတွေအဖြစ် ယူဆပြီး အလုပ်လုပ်မှာမို့လို့။  

In [ ]:
!mv train.my.clean train.my

In [ ]:
!mv dev.my.clean dev.my

In [ ]:
!mv test.my.clean test.my

In [ ]:
!tree .

## Moses SMT Framework

အောက်ပါ လင့်တွေကနေ လေ့လာပါ။ ထဲထဲဝင်ဝင် သိချင်ရင်တော့ user manual မှာ လက်တွေ့ ဘယ်လို run ရမယ် ဆိုတာ အသေးစိတ် ရေးထားတာမို့ ဖတ်ရှုလေ့လာကြပါ။ SMT ထဲက မော်ဒယ်တွေ၊ လက်တွေ့လုပ်ပုံတွေ နားလည်ရင် အလုပ်ထဲမှာ ဖြေရှင်းကြရမယ့် အင်ဂျင်နီယာကိစ္စတွေအတွက် အများကြီး အသုံးဝင်ပါလိမ့်မယ်။   

1. homepage: [https://www2.statmt.org/moses/](https://www2.statmt.org/moses/)
2. User Manual: [http://www2.statmt.org/moses/manual/manual.pdf](http://www2.statmt.org/moses/manual/manual.pdf)

ဆရာက moses framework ရဲ့ executable binary ဖိုင်တွေနဲ့ တခြား လိုအပ်တဲ့ script တွေကို အောက်ပါ path မှာ ပြင်ဆင်ထားပါတယ်။  

In [ ]:
!ls /home/ye/tool/mosesbin

In [ ]:
%cd /home/ye/tool/mosesbin

moses framework အထဲမှာ ပါဝင်တဲ့ ဖိုင်တွေနဲ့ သူ့ရဲ့ folder structure ကို အကြမ်းဖျဉ်း လေ့လာကြရအောင်။  

In [ ]:
%cd ubuntu-17.04/

In [ ]:
!tree .

## Phrase Based SMT Tutorial  

YTU မှာ ဆရာ workshop လုပ်တုန်းက တင်ပေးထားခဲ့တဲ့ configuration ဖိုင်စတာတွေက အသုံးဝင်ပါလိမ့်မယ်။   

**MTRSS: Machine Translation Research Summer School**

Here, I uploaded some files that I used at MTRSS (8-10 Oct, 2019),
Yangon Technological University (YTU), Yangon, Myanmar.

Link: [https://github.com/ye-kyaw-thu/MTRSS/tree/master/pbsmt](https://github.com/ye-kyaw-thu/MTRSS/tree/master/pbsmt)  

ဒီကနေ စပြီး လက်တွေ့ PBSMT ကို run ဖို့ စပြင်ပြီး step-by-step လုပ်ပြသွားမယ်။  

In [ ]:
%pwd

## Preparing Testset SGM Files 

ခု လက်တွေ့လုပ်ပြမယ့် SMT က moses2 ရဲ့ experimental system ပုံစံမျိုး configuration file ဆောက်ပြီးသွားမှာမို့လို့ SGM ဖိုင်ကို ရှိမရှိ စစ်တယ်။ မတွေ့ရင် run မလုပ်ပေးဘူး။ အဲဒီအတွက် test data တွေကို SGM file format အရင် ပြောင်းကြရအောင်။  

ဆရာပြင်ပေးထားတဲ့ perl script သုံးဖိုင်က အောက်ပါ path အောက်မှာရှိတယ်။ ဒီ Notebook နဲ့ အတူလည်း ရှဲပေးထားပါမယ်။  

[https://github.com/ye-kyaw-thu/MTRSS/tree/master/pbsmt/data/test-sgm](https://github.com/ye-kyaw-thu/MTRSS/tree/master/pbsmt/data/test-sgm)

အောက်က cell မှာ အဲဒီ script သုံးဖိုင်ကို GitHub ကနေ တိုက်ရိုက် download ဆွဲပြီး သုံးပါမယ် (ဖိုင်တွေက Myanmar–Rakhine အတွက် ပြင်ပြီးသား ဖြစ်လို့ ဒီအတိုင်း သုံးလို့ရပါတယ်)။


In [ ]:
# Simple way: download the 3 Perl scripts directly from Sayar Ye Kyaw Thu's MTRSS repo
!mkdir -p /home/ye/exp/SMT-NMT_tutorial/clean-data/scripts
%cd /home/ye/exp/SMT-NMT_tutorial/clean-data/scripts

base = "https://raw.githubusercontent.com/ye-kyaw-thu/MTRSS/master/pbsmt/data/test-sgm"
!wget -q -O generate_sgms.pl $base/generate_sgms.pl
!wget -q -O src2sgm.pl $base/src2sgm.pl
!wget -q -O ref2sgm.pl $base/ref2sgm.pl
!chmod +x *.pl
!ls -l


In [ ]:
!chmod +x /home/ye/exp/SMT-NMT_tutorial/clean-data/scripts/*.pl


In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/clean-data/scripts/

In [ ]:
!cat generate_sgms.pl

**အထက်က perl script က ပြင်စရာ မလိုပါ။ အောက်က perl script နှစ်ဖိုင်ကလည်း MTRSS repo ထဲမှာကတည်းက Myanmar–Rakhine experiment အတွက် ရေးထားပြီးသားမို့ ဒီတစ်ခါ ဝင်ပြင်စရာ မလိုတော့ပါ။ ဘာတွေ ပါဝင်လဲ ဝင်ဖတ်ကြည့်ရအောင်။**


In [ ]:
!cat ./src2sgm.pl

**နောက်ထပ် တစ်ဖိုင်က reference အတွက်ပါ။** 

In [ ]:
!cat ./ref2sgm.pl

### Run perl script  

generate လုပ်ပေးမယ့် perl ကို run လိုက်ရင်အဲဒီ ဖိုလ်ဒါ path အောက်မှာပဲ SGM ဖိုင်တွေ ထွက်လာလိမ့်မယ်။  

In [ ]:
# Safety check: generate_sgms.pl silently does NOTHING if ../train.* is missing.
# Make sure clean-data/ has all 6 corpus files; restore any missing one
# (cleaned version from normalize/ if it exists, otherwise the raw data/ copy).
import os, shutil
CD, DATA = "/home/ye/exp/SMT-NMT_tutorial/clean-data", "/home/ye/exp/SMT-NMT_tutorial/data"
for split in ["train", "dev", "test"]:
    for lang in ["my", "rk"]:
        dst = f"{CD}/{split}.{lang}"
        if not os.path.exists(dst):
            for src in [f"{CD}/normalize/{split}.{lang}.clean",
                        f"{CD}/normalize/{split}.{lang}",
                        f"{DATA}/{split}.{lang}"]:
                if os.path.exists(src):
                    shutil.copy(src, dst)
                    print(f"restored {split}.{lang}  <-  {src}")
                    break
print("clean-data now has:", sorted(f for f in os.listdir(CD) if "." in f))


In [ ]:
!perl ./generate_sgms.pl

In [ ]:
!ls *sgm

In [ ]:
# Stage the SGM files where the Moses config expects them:
# [EVALUATION:test] reads $myrk-data/test-sgm/test.*.src.sgm / .ref.sgm
!mkdir -p ../test-sgm
!cp -f ./*.sgm ../test-sgm/
!ls ../test-sgm


In [ ]:
!head test.my.ref.sgm

In [ ]:
!head test.rk.ref.sgm

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial

## GIZA++ Installation  

Alignment လုပ်ပေးမယ့် tool က ကိုယ့်စက်ထဲမှာ မရှိသေးရင် git clone လုပ်ပြီး install လုပ်ပါ။ ပြီးရင် အဲဒီ path ကို config ဖိုင်မှာ ဝင်ဖြည့်ပေးပါ။  


git clone https://github.com/moses-smt/giza-pp လုပ်ပြီး အဲဒီ path အောက်ကို ဝင်ပြီး make all command ကို run ပါ။  
ye@lst-hpc3090:~/tool/giza-pp$ make all
...
...
...
g++  -O3 -W -Wall plain2snt.cpp -o plain2snt.out
g++  -O3 -g -W -Wall snt2cooc.cpp -o snt2cooc.out
make[1]: Leaving directory '/home/ye/tool/giza-pp/GIZA++-v2'
make -C mkcls-v2
make[1]: Entering directory '/home/ye/tool/giza-pp/mkcls-v2'
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c GDAOptimization.cpp -o GDAOptimization.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c HCOptimization.cpp -o HCOptimization.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c Problem.cpp -o Problem.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c IterOptimization.cpp -o IterOptimization.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c ProblemTest.cpp -o ProblemTest.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c RRTOptimization.cpp -o RRTOptimization.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c MYOptimization.cpp -o MYOptimization.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c SAOptimization.cpp -o SAOptimization.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c TAOptimization.cpp -o TAOptimization.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c Optimization.cpp -o Optimization.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c KategProblemTest.cpp -o KategProblemTest.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c KategProblemKBC.cpp -o KategProblemKBC.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c KategProblemWBC.cpp -o KategProblemWBC.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c KategProblem.cpp -o KategProblem.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c StatVar.cpp -o StatVar.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c general.cpp -o general.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c mkcls.cpp -o mkcls.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -o mkcls GDAOptimization.o HCOptimization.o Problem.o IterOptimization.o ProblemTest.o RRTOptimization.o MYOptimization.o SAOptimization.o TAOptimization.o Optimization.o KategProblemTest.o KategProblemKBC.o KategProblemWBC.o KategProblem.o StatVar.o general.o mkcls.o
make[1]: Leaving directory '/home/ye/tool/giza-pp/mkcls-v2'
ye@lst-hpc3090:~/tool/giza-pp$

**PBSMT ကို ဖိုလ်ဒါ အသစ်တစ်ခုဆောက်ပြီး အဲဒီအောက်မှာပဲ run သွားမယ်။**

In [ ]:
!mkdir pbsmt

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/pbsmt

**kc-my.config.baseline** ကို လေ့လာရင်း အဲဒီဖိုင်ကို ရခိုင်-မြန်မာ ဘာသာပြန်ဖို့အတွက် ဝင်ပြင်ကြရအောင်။  

In [ ]:
!cat kc-my.config.baseline

Original config.baseline ကို ဝင်ပြင်ပါ။ ပြီးရင် အောက်ပါအတိုင်း run ပြီး Rakhine-Myanmar အတွက် config ဖိုင်ကို generate လုပ်ပါ။  

In [ ]:
!ls

**ပြင်ပြီးသွားတဲ့ config.baseline ဖိုင်က အောက်ပါအတိုင်းပါ**

In [ ]:
!cat config.baseline

## Updating generate_config.pl

generate_config.pl ဖိုင်ကို ဝင်ပြင်ရအုံးမယ်။  

In [ ]:
!cat generate_config.pl

**After updating**

Update လုပ်ပြီးသားဖိုင်က အောက်ပါအတိုင်းပါ။  


In [ ]:
!cat generate_config.pl

**အဓိက ဝင်ပြင်ခဲ့တာက path တွေပါပဲ**
အထက်ပါအတိုင်း ပြင်ဆင်ပြီးသွားရင် configuration ဖိုင်ကို generate လုပ်ပေးမယ့် perl script ကို run ကြရအောင်။  

In [ ]:
 !ls

In [ ]:
!perl ./generate_config.pl

In [ ]:
!ls

In [ ]:
!ls -ad */

In [ ]:
%cd ..

In [ ]:
!tree .

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/pbsmt/baseline/rk-my/


In [ ]:
!cat ./config.baseline.rk-my


## show updated perl

## Let's DO SMT

In [ ]:
%pwd

In [ ]:
!time perl ./run-baseline.pl

## ERROR

In [ ]:
!cat ./run1.log

## Found Error

run-baseline.pl

        chomp $config;

        my @toks = split /\./, $config;
        my $jobName = "$toks[$#toks]-$toks[$#toks - 1]";

        print "$jobName $config\n";

        `/home/lar/tool/moses/scripts/ems/experiment.perl -config $config -exec -no-graph 2>&1 | tee -a >
        #`/home/ros/mosesdecoder/scripts/ems/experiment.perl -config $config -exec -no-graph`;


## Run Again  

In [ ]:
%pwd

In [ ]:
!time perl ./run-baseline.pl

## We Forgot!

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/pbsmt

In [ ]:
%pwd

In [ ]:
!perl ./generate_configs.pl

In [ ]:
!time perl ./run-baseline.pl

## One More Error Example

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/pbsmt/baseline/rk-my/steps/1

In [ ]:
!cat TRAINING_prepare-data.1.STDERR

## To Add graph1.png 

In [ ]:
from IPython.display import Image
Image(filename='/home/ye/exp/SMT-NMT_tutorial/error-examples/GIZA-error.png', width=800)

## Solution

အထက်ပါ error ကိုတော့ အတွေ့အကြုံအရ သိနေတာမို့ mkcls binary ဖိုင်ကို GIZA++-v2/ ဖိုလ်ဒါအောက်ကို ကော်ပီကူးထည့်ပေးလိုက်ယုံနဲ့ ပြေလည်ပါတယ်။ အောက်ပါအတိုင်းပါ။  

ye@lst-hpc3090:~/tool/giza-pp$ cp ./mkcls-v2/mkcls ./GIZA++-v2/  

## ERROR

တချို့ error တွေက terminal မှာ ဘာမှမပြပဲ ရပ်သွားတာမျိုးလည်း ဖြစ်တတ်ပါတယ်။ အဲဒါဆိုရင် အများသောအားဖြင့် perl script မှာ path ပေးတာ လွဲသွားလို့ပါ။  

ဘာဖိုင်မှ ထွက်မလာတာကို တမင်တကာ လုပ်ပြထားတာပါ။   
config နဲ့ SMT ကို run မယ်ဆိုရင် နောက်ထပ် ပြင်ရမယ့် အဆင့် ရှိသေးတယ်။ အဲဒါက ဒေတာဖိုလ်ဒါအောက်မှာ ဝင်လုပ်ရမယ်။  

Refer following link:  
[https://github.com/ye-kyaw-thu/MTRSS/tree/master/pbsmt/data](https://github.com/ye-kyaw-thu/MTRSS/tree/master/pbsmt/data)  

ဒီတစ်ခေါက် tutorial အတွက်တော့ SMT-NMT_tutorial/pbsmt/ ဖိုလ်ဒါအောက်မှာလည်း ကောပီကူးထည့်ပေးထားမယ်။  
အဲဒီ script ကို ကိုယ့် စက်ထဲက parallel data ဒေတာ ဖိုလ်ဒါအောက်ကို ကောပီကူးယူပြီး ပြင်ဆင်ကြပါ။  

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial


In [ ]:
!ls ./scripts

In [ ]:
!cp -r scripts/ ./clean-data/


path ပြင်ပြီးသား update လုပ်ပြီးသား perl script တွေနဲ့ run ကြရလိမ့်မယ်။ ဥပမာ အောက်ပါအတိုင်း  

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/clean-data/

In [ ]:
%cd scripts/

In [ ]:
!tree .

In [ ]:
!perl ./generate_sgms.pl

**run-pbsmt.sh ကို Executable ဖြစ်အောင် permission ပေးကြရအောင်...**

In [ ]:
!chmod +x ./run-pbsmt.sh

In [ ]:
!cat run-pbsmt.sh

**Shell script ကို run မယ်**

In [ ]:
!./run-pbsmt.sh

## Notes

ဒီ notebook မှာ ကြုံခဲ့ရသလိုပါပဲ error အများစုက path လွဲတာ၊ config ဖိုင်မှာ တခုခု update လုပ်ဖို့ ကျန်တာ၊ test data တွေကို SGM format ပြောင်းဖို့ မေ့ကျန်ခဲ့တာ၊ GIZA++ ရဲ့ mkcls ကို ရှာမတွေ့တာ တွေပါပဲ။ အဲဒီလိုမျိုး error မှ မဟုတ်ရင်တော့ ဒေတာနဲ့ ဆိုင်ပါလိမ့်မယ်။ Parallel corpus မှာ တချို့ blank line တွေ ပါနေတာတို့ special character တွေကို ရှင်းဖို့ လိုနေသေးတာတို့ ဖြစ်ဖို့ များပါတယ်။  

Error အကုန် ရှင်းပြီးရင်တော့ run လိုက်ရင် အစအဆုံး ပြီးအောင် run ပေးသွားပါလိမ့်မယ်။  

## PBSMT Experiment 

In [ ]:
%pwd

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/pbsmt/


In [ ]:
!ls *

In [ ]:
# generate_configs.pl deliberately dies if baseline/ already exists
# (so it never overwrites a finished experiment) — clear any earlier
# config-only baseline/ before regenerating.
!rm -rf ./baseline
# make sure evaluation sgm files are staged (generated in the SGM section)
!mkdir -p /home/ye/exp/SMT-NMT_tutorial/clean-data/test-sgm
!cp -f /home/ye/exp/SMT-NMT_tutorial/clean-data/scripts/*.sgm /home/ye/exp/SMT-NMT_tutorial/clean-data/test-sgm/ 2>/dev/null
!perl ./generate_configs.pl


In [ ]:
!ls * --color=auto

In [ ]:
!tree ./baseline

In [ ]:
!time perl run-baseline.pl

Training data က စာကြောင်းရေ ၅၀၀၀ ကျော်၊ development data က စာကြောင်းရေ ၅၀၀ ကျော်နဲ့ test data က စာကြောင်းရေ ၁၀၀ နဲ့ပဲ run တာမို့လို့ မြန်မာ-ရခိုင်၊ ရခိုင်-မြန်မာ အတွက်က ဆာဗာပေါ်မှာ မိနစ် ၂၀ကျော်ပဲ ကြာပါတယ်။  

System တစ်ခုလုံး run သွားတဲ့ ဂရဖ်တွေကို ကြည့်ကြည့်ရအောင်။  

အောက်ပါ ဂရဖ်က မြန်မာ-ရခိုင် အတွက်ပါ။

> **မှတ်ချက် (Colab ဗားရှင်း):** `run-baseline.pl` က `experiment.perl` ကို `-no-graph` option နဲ့ ခေါ်ထားတာမို့ (upstream MTRSS script အတိုင်းပါပဲ) training အတွင်းမှာ `graph.1.png` ထုတ်မပေးပါဘူး။ Graph က pipeline အဆင့်တွေကို မြင်သာအောင် ဆွဲပြတာ သက်သက်မို့ မရှိလည်း model နဲ့ BLEU ရလဒ်တွေအပေါ် လုံးဝ သက်ရောက်မှု မရှိပါဘူး။ အောက်က cell က graph `.dot` ဖိုင် ရှိခဲ့ရင် graphviz နဲ့ render လုပ်ပေးပြီး၊ မရှိရင် ကျော်သွားလို့ ရကြောင်း ပြောပါလိမ့်မယ်။


In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/pbsmt/baseline/my-rk/steps/1

In [ ]:
!ls *

In [ ]:
# run-baseline.pl uses "experiment.perl -no-graph" (same as upstream MTRSS),
# so training renders no graph.1.png. If EMS left a graph .dot description,
# render it with graphviz; otherwise skip — the graph is only a picture of
# the pipeline steps and does not affect the model or the BLEU score.
import glob, subprocess
dot_files = [f for f in glob.glob("graph.*") if not f.endswith(".png")]
if dot_files:
    subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], capture_output=True)
    for d in dot_files:
        subprocess.run(["dot", "-Tpng", d, "-o", d + ".png"])
    print("rendered:", sorted(glob.glob("*.png")))
else:
    print("No graph files here (runner used -no-graph) — skip to the evaluation cells below.")


graph.1.png ကိုပဲ quality ကောင်းအောင် ပုံပြင်ထုတ်ထားပြီး error-examples/ ဖိုလ်ဒါအောက်ထဲမှာပဲ သိမ်းထားပေးလိုက်ပါတယ်။  

In [ ]:
from IPython.display import Image
Image(filename='/home/ye/exp/SMT-NMT_tutorial/error-examples/graph.1.my-rk.png', width=800)

အောက်ဆုံးပိုင်းမှာ အနီနဲ့ nist-bleu score နဲ့ ပတ်သက်ပြီး ပြနေပေမဲ့ စိုးရိမ်စရာ မရှိပါဘူး။ 
ဒါပေမဲ့ multi-bleu နဲ့ တွက်ထားတာကိုတော့ ဆရာတို့ လိုချင်တယ်။ အဲဒါက ပါမလာဘူး။ 
config ဖိုင်ထဲမှာ assign မလုပ်ထားလို့ ပေးနေတဲ့ error လို့ ထင်ပါတယ်။ 

Config ဖိုင် template ကို ဝင်ကြည့်တော့ အောက်ပါအတိုင်း comment ပိတ်ထားတာကို တွေ့ရတယ်။
သေချာတယ်။ ကျောင်းသားတွေကို ကိုယ်ဖာသာကိုယ် ရှာတွက်တတ်ဖို့ exercise လုပ်ဖို့ ပိတ်ထားခဲ့တာ။  

```
### BLEU
#
nist-bleu = $moses-script-dir/generic/mteval-v13a.pl
nist-bleu-c = "$moses-script-dir/generic/mteval-v13a.pl -c"
#multi-bleu = $moses-script-dir/generic/multi-bleu.perl
#ibm-bleu =
```

အထက်က multi-bleu လိုင်းကိုပါ ဖွင့်ပေးပြီး နောက်တစ်ခေါက် ထပ် run ကြည့်ရအောင်။  

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/pbsmt

In [ ]:
%rm -r baseline/

In [ ]:
!perl ./generate_configs.pl

In [ ]:
!tree ./baseline

config ဖိုင်ကို ပြင်ပြီး နောက်တစ်ခေါက် ထပ် run မယ်။  

In [ ]:
!time perl ./run-baseline.pl

test-run လုပ်ထားတဲ့ my-rk ရလဒ်ကို ဝင်လေ့လာရအောင်။  

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/pbsmt/baseline/my-rk/evaluation

In [ ]:
!ls * --color=auto

In [ ]:
!cat test.multi-bleu.1

BLEU score က test-run လုပ်ထားတာမို့ တအားနည်းပါတယ်။ SMT pipeline အစအဆုံး ပြီးမပြီး confirm လုပ်ရအောင်။  

In [ ]:
from IPython.display import Image
Image(filename='/home/ye/exp/SMT-NMT_tutorial/error-examples/graph.1.my-rk-2ndtime.png', width=800)

အောက်ဆုံးပိုင်းမှာ multi-bleu ပါလာတာကို တွေ့ရပါပြီ။ အဲဒါကြောင့် BLEU score တွက်ပေးထားတာကို မြင်ရတာပါ။ ကောင်းပြီ။ ရခိုင်-မြန်မာ ဘက်ရဲ့ ရလဒ်ကို ဝင်ကြည့်ရအောင်။  

In [ ]:
%cd /home/ye/exp/SMT-NMT_tutorial/pbsmt/baseline/rk-my/evaluation

In [ ]:
!cat ./test.multi-bleu.1

ရခိုင်-မြန်မာ ရလဒ်ကတော့ အထက်ပါအတိုင်းပါပဲ။

> **မှတ်ချက် (Colab / Rakhine ဗားရှင်း):** ဆရာ့မူရင်း Myanmar–Thai corpus နဲ့ run တုန်းက
> BLEU **11.77** ရခဲ့ပါတယ်။ အခု Myanmar–Rakhine corpus နဲ့ဆိုရင် score က ကွာပါလိမ့်မယ် —
> မြန်မာနဲ့ ရခိုင်က အလွန်နီးစပ်တဲ့ ဘာသာစကားတွေဖြစ်လို့ BLEU က သိသိသာသာ **ပိုမြင့်** တတ်ပါတယ်။
> Pipeline (align → phrase table → LM → tune → decode → BLEU) ကတော့ တစ်ထပ်တည်း အတူတူပါပဲ။

test-run အစအဆုံး ပြီးပါပြီ။

ခုလောက်ဆိုရင် SMT running pipeline တော့ အကြမ်း နားလည်ပြီလို့ ယူဆပါတယ်။
(မူရင်း သင်တန်းအစီအစဉ်အရ နောက်ထပ် Jupyter notebook နှစ်ခုမှာ မြန်မာ-ထိုင်း medical domain
parallel corpus အပြည့်အစုံနဲ့ ထပ် run ဖို့ ဖြစ်ပါတယ် — အဲဒီ corpus က public မဟုတ်သေးလို့
ဒီ Colab ဗားရှင်းမှာ Rakhine corpus ကို အစားထိုး သုံးထားတာပါ။)
